# E00B — Budget, Token & Runtime Forecast

Reconstruction-v2 analysis notebook. Local-only: reads `results/budget/*.csv`/`*.json`
(already computed by `scripts/build_e00b_forecast.py`) and imports `evaluation.budget` for the
reusable pricing/cost/budget-gate functions. **Zero LLM/API calls, zero local-model
inference.** The only network activity anywhere in E00B was a documentation pricing lookup,
not a model call.

See `README.md` and `summary.md` in this directory for the full write-up and the 10 required
decision answers.

In [1]:
import csv
import json
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parents[1] if Path.cwd().name == "E00B_budget_forecast" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from evaluation.budget import check_budget, estimate_cost, load_pricing

BUDGET = REPO_ROOT / "results/budget"
print("repo root:", REPO_ROOT)

repo root: /Users/asmitha/Documents/course/PE6201-EMERGING AI TECHNOLOGIES/Project/ndatrace


## 1. Planning budget

In [2]:
plan = json.load(open(BUDGET / "budget_plan.json"))
print("planning_budget_usd:", plan["planning_budget_usd"])
print("source:", plan["planning_budget_source"], "| recorded:", plan["date_recorded"])
print("historical .env MAX_BUDGET_USD (stale, not trusted):", plan["historical_env_value_MAX_BUDGET_USD"])
print("protected reserve (25%):", plan["protected_reserve_usd"])
print("allowed forward-spend budget:", plan["allowed_budget_usd"])

planning_budget_usd: 5.0
source: user-reported current balance | recorded: 2026-09-26
historical .env MAX_BUDGET_USD (stale, not trusted): 6.99
protected reserve (25%): 1.25
allowed forward-spend budget: 3.75


## 2. Historical spend ledger (real, audited local files)

In [3]:
with open(BUDGET / "historical_spend.csv") as f:
    rows = list(csv.DictReader(f))
print(f"{len(rows)} historical runs found (results/runs/ + results/archive/runs/)")

hosted = [r for r in rows if r["provider"] == "openrouter"]
total = sum(float(r["reported_cost_usd"]) for r in hosted if r["reported_cost_usd"] not in ("", "UNKNOWN"))
print(f"hosted (openrouter) runs: {len(hosted)}, total historical spend: ${total:.4f}")
print(f"(this is history for audit purposes only -- NOT subtracted from the $5 forward budget)")

unknown = [r for r in rows if r["reported_cost_usd"] == "UNKNOWN"]
print(f"rows with unknown cost (marked explicitly, not fabricated): {len(unknown)}")
for r in unknown:
    print(" -", r["experiment_id"], "--", r["notes"][:80])

27 historical runs found (results/runs/ + results/archive/runs/)
hosted (openrouter) runs: 19, total historical spend: $3.0011
(this is history for audit purposes only -- NOT subtracted from the $5 forward budget)
rows with unknown cost (marked explicitly, not fabricated): 1
 - checkpoint_T041_final_test_rag_agent_llama3.2_3b -- split=? sample_size=? architecture=? n_cases=? file=results/archive/runs/pre_c1_


## 3. Current verified pricing

Shows PUBLISHED price (the model's real rate card) vs. EFFECTIVE cost assumption (what
forecasts actually use -- conservative by default, never assumed $0 just because a free tier
exists). Corrected this revision: GPT-5 mini's cached-input rate ($0.025/M, was mis-recorded
as $0.03/M), Groq gpt-oss-20b's canonical price (was wrongly $0/token; now $0.075/$0.30 per
M, with the free tier recorded as a separate rate-limited account condition), and Gemini's
retirement note (now provider-specific, not a blanket claim).

In [4]:
with open(BUDGET / "current_pricing.csv") as f:
    for row in csv.DictReader(f):
        print(f"{row['provider']:12s} {row['model']:35s} "
              f"published: in=${row['published_input_price_per_million']}/M out=${row['published_output_price_per_million']}/M  "
              f"| effective: in=${row['effective_cost_assumption_input_per_million']}/M out=${row['effective_cost_assumption_output_per_million']}/M  "
              f"| tier={row['account_tier']}")
        if row["lifecycle_status"]:
            print(f"    lifecycle: {row['lifecycle_status']}")

groq         openai/gpt-oss-20b                  published: in=$0.075/M out=$0.3/M  | effective: in=$0.075/M out=$0.3/M  | tier=free (rate-limited) -- pay-as-you-go paid tier also available at the published rate
    lifecycle: no announced retirement found
local/ollama llama3.2:3b                         published: in=$0.0/M out=$0.0/M  | effective: in=$0.0/M out=$0.0/M  | tier=local compute -- no third-party billing account exists at all (not a free tier of a paid service)
    lifecycle: n/a -- local, not a hosted service with a lifecycle
openrouter   google/gemini-2.5-flash-lite        published: in=$0.1/M out=$0.4/M  | effective: in=$0.1/M out=$0.4/M  | tier=pay-as-you-go
    lifecycle: provider-specific -- NOT a blanket retirement, corrected this pass
openrouter   openai/gpt-5-mini                   published: in=$0.25/M out=$2.0/M  | effective: in=$0.25/M out=$2.0/M  | tier=pay-as-you-go
    lifecycle: no announced retirement found


## 4. Token estimates (empirical, from real TRAIN-split text)

In [5]:
with open(BUDGET / "token_estimates.csv") as f:
    for row in csv.DictReader(f):
        print(f"{row['quantity']:40s} {row['value']:>10s}  {row['unit']}")

system_prompt_tokens                            571  tokens (cl100k_base approx)
hypothesis_tokens_mean                         15.6  tokens
gold_evidence_tokens_mean                      90.7  tokens (TRAIN, E/C cases)
gold_evidence_tokens_p90                        170  tokens
gold_evidence_tokens_max                        603  tokens
oracle_input_tokens_mean                      677.2  tokens (estimate)
oracle_input_tokens_p90                         769  tokens (estimate)
output_tokens_compact                            17  tokens
output_tokens_verbose                            75  tokens


## 5. Compact vs. verbose output — cost difference

In [6]:
cvv = plan["compact_vs_verbose"]
print("Instructor-cited figures (644in/449out/$0.001059/9.21s):")
print(" ", cvv["instructor_cited_figures"]["verification_result"])
print()
print("This project's own real measured data (T024 RAG, Gemini, v2 prompt):")
own = cvv["this_projects_own_real_measured_data"]
print(f"  mean input={own['mean_input_tokens']} mean output={own['mean_output_tokens']} "
      f"mean cost=${own['mean_cost_usd']}")
print()
print("Oracle scenario, compact vs verbose output, per-case cost:")
scen = cvv["oracle_scenario_compact_vs_verbose"]
print(f"  Gemini:    compact=${scen['gemini_compact_usd_per_case']}  verbose=${scen['gemini_verbose_usd_per_case']}")
print(f"  GPT-5-mini: compact=${scen['gpt5mini_compact_usd_per_case']}  verbose=${scen['gpt5mini_verbose_usd_per_case']}")
print()
print("Design rule:", cvv["design_rule"])

Instructor-cited figures (644in/449out/$0.001059/9.21s):
  These are the INSTRUCTOR's estimate, based on a GPT-5-mini example from the Week 3 submission (~644 input / ~449 output tokens, ~$0.001059/9.21s per case) -- NOT this project's own measured NDATrace data. Not found anywhere in this project's own result files under those exact values. Reported here as correctly attributed, not treated as an NDATrace project fact.

This project's own real measured data (T024 RAG, Gemini, v2 prompt):
  mean input=853.6 mean output=116.1 mean cost=$0.0001318

Oracle scenario, compact vs verbose output, per-case cost:
  Gemini:    compact=$7.45e-05  verbose=$9.77e-05
  GPT-5-mini: compact=$0.0002033  verbose=$0.0003193

Design rule: Bulk evaluation outputs should default to minimal structured JSON (label + evidence IDs only). Verbose natural-language explanations should only be generated in experiments that specifically evaluate explanation/faithfulness quality (axis D of docs/project_contract.md se

## 6. Oracle cost scenarios (real historical-cost basis)

In [7]:
with open(BUDGET / "experiment_forecast.csv") as f:
    for row in csv.DictReader(f):
        if row["experiment"] == "E01_Oracle":
            print(f"n={row['n_cases']:>5s}  hosted_models={row['hosted_models']}  "
                  f"estimated_cost=${row['estimated_cost_usd']}")

n=  100  hosted_models=2  estimated_cost=$0.1116
n=  150  hosted_models=2  estimated_cost=$0.1674
n=  300  hosted_models=2  estimated_cost=$0.3348
n=  500  hosted_models=2  estimated_cost=$0.558
n= 1000  hosted_models=2  estimated_cost=$1.116


## 7. Later hosted-experiment forecast (E03, E15) and cumulative scenarios

In [8]:
for name, scen in plan["scenarios"].items():
    print(f"--- {name} ---")
    print(" ", scen["description"])
    cost = scen.get("estimated_total_cost_usd", scen.get("estimated_cost_usd"))
    print(f"  estimated cost: ${cost}   within allowed budget (${plan['allowed_budget_usd']}): {scen['within_allowed_budget']}")
    print()

--- LEAN ---
  Oracle n=150 (2 hosted models) + a small 150-case E15 spot-check on one hosted model only, half the E03 exploration deferred to base-case
  estimated cost: $0.3001   within allowed budget ($3.75): True

--- RECOMMENDED ---
  Oracle n=300 (2 hosted models) + E03 prompt selection (150-case TRAIN subset x 4 variants, 1 hosted model) + E15 (300-case stratified TEST subsample x 3 architectures, 1 hosted model) + 10% rerun reserve
  estimated cost: $0.7472   within allowed budget ($3.75): True

--- MAXIMUM_SAFE ---
  Oracle n=500 (2 hosted models) + E03 (300-case TRAIN subset x 4 variants) + E15 (500-case stratified TEST subsample x 3 architectures) -- still preserves the 25% reserve, never spends the full $5
  estimated cost: $1.1586   within allowed budget ($3.75): True



## 7b. Gemini lifecycle — provider-specific, not a blanket claim

Corrected this revision: Google's own Gemini Developer API (AI Studio) lists
`gemini-2.5-flash-lite` as stable with no announced shutdown. Google Cloud Vertex AI
separately lists an 2026-10-16 retirement. NDATrace calls OpenRouter without pinning a
backend provider, and OpenRouter serves this model via BOTH Vertex AI and Google AI Studio
with automatic failover -- so the risk is real but not certain, and applies to the Vertex
backend specifically, not to the model universally.

In [9]:
import yaml
gemini_cfg = yaml.safe_load(open(REPO_ROOT / "configs/pricing/openrouter_google_gemini-2.5-flash-lite.yaml"))
for k, v in gemini_cfg["lifecycle"].items():
    print(f"{k}:\n  {v}\n")

status:
  provider-specific -- NOT a blanket retirement, corrected this pass

backend_used_by_ndatrace:
  OpenRouter's generic chat-completions endpoint (pipeline/model_gateway.py, base_url=settings.openrouter_base_url), with NO provider pinned in code.

ndatrace_risk_detail:
  OpenRouter serves google/gemini-2.5-flash-lite via 2 backend providers with automatic failover: Google Vertex AI and Google AI Studio (confirmed via web search 2026-09-26). Since NDATrace's OpenRouter calls never pin a provider, any given call could be routed through either backend.

google_ai_studio_gemini_developer_api:
  Per Google's own Gemini API documentation, gemini-2.5-flash-lite is stable, NOT deprecated, with no announced shutdown date (checked 2026-09-26).

google_cloud_vertex_ai:
  Google Cloud separately lists Gemini 2.5 Flash-Lite retirement on Vertex AI as 2026-10-16 (~3 weeks from today, checked 2026-09-26).

conclusion:
  The 2026-10-16 retirement applies to the Vertex AI backend specifically, n

## 8. Hard budget gate demo + running reconstruction-v2 spend ledger (no network call)

In [10]:
# Demonstrates the reusable evaluation.budget.check_budget() gate any future hosted
# reconstruction-v2 script should call before spending real money.
ok = check_budget(projected_experiment_cost_usd=0.7472, actual_spend_so_far_usd=0.0,
                   planning_budget_usd=5.0, protected_reserve_fraction=0.25)
print(ok.reason)

blocked = check_budget(projected_experiment_cost_usd=4.5, actual_spend_so_far_usd=0.0,
                        planning_budget_usd=5.0, protected_reserve_fraction=0.25)
print(blocked.reason)

print()
ledger_status = plan["reconstruction_v2_spend_ledger"]
print("Running reconstruction-v2 spend ledger:", ledger_status["path"])
print(" status:", ledger_status["status"])
print(" current total: $", ledger_status["current_total_usd"])
print(" (this is SEPARATE from historical_spend.csv -- never double-counted)")

OK: projected total $0.7472 <= allowed $3.7500 (planning budget $5.00 - reserve $1.25)
BLOCKED: projected total $4.5000 > allowed $3.7500 (planning budget $5.00 - reserve $1.25)

Running reconstruction-v2 spend ledger: results/budget/reconstruction_spend_ledger.csv
 status: READY -- append-only, currently empty (no reconstruction-v2 hosted call has been made yet)
 current total: $ 0.0
 (this is SEPARATE from historical_spend.csv -- never double-counted)


## 9. Runtime scenarios (local + hosted, real measured per-case latency)

In [11]:
with open(BUDGET / "runtime_forecast.csv") as f:
    for row in csv.DictReader(f):
        print(f"{row['scenario']:35s} {row['run']:55s} cases={row['cases']:>5s} "
              f"sec/case={row['sec_per_case']:>6s}  ~{row['estimated_hours']:>6s}h")

BASE-CASE (measured)                A1 full-context, local Llama                            cases= 2091 sec/case=  7.68  ~  4.46h
BASE-CASE (measured)                A2 RAG, local Llama                                     cases= 2091 sec/case=  4.53  ~  2.63h
BASE-CASE (measured)                A3 RAG+agent (measured, already includes historical escalation mix), local Llama cases= 2091 sec/case=  9.19  ~  5.34h
BEST-CASE                           A3 RAG+agent local, escalation_rate=0.3, extra_turn_factor=1.0 cases= 2091 sec/case=  4.53  ~  2.63h
BASE-CASE                           A3 RAG+agent local, escalation_rate=0.45, extra_turn_factor=1.3 cases= 2091 sec/case=  5.76  ~  3.34h
WORST-CASE                          A3 RAG+agent local, escalation_rate=0.6, extra_turn_factor=1.8 cases= 2091 sec/case=  8.89  ~  5.16h
E15 subsample (measured per-case)   A1 full-context, hosted Gemini                          cases=  300 sec/case=  1.11  ~ 0.093h
E15 subsample (measured per-case)   A1 full

## 10. Recommended experimental budget plan — the 10 required answers

In [12]:
for k, v in plan["answers"].items():
    print(f"{k}:")
    if isinstance(v, list):
        for item in v:
            print("   -", item)
    else:
        print("  ", v)
    print()

A_hosted_budget_available_usd:
   5.0

B_protected_reserve_usd:
   1.25

C_E01_oracle_budget_envelope_usd:
   up to ~$0.56 (n=500, 2 hosted models, historical-cost basis) -- Oracle is NOT the binding constraint at any realistic sample size

D_feasible_oracle_subset_size:
   150-300 cases recommended (matches the historical convention for direct comparability, and is far below any budget constraint)

E_can_2_hosted_oracle_models_fit:
   True

F_budget_remaining_for_E03_E15_after_oracle_usd:
   3.4152

G_recommended_bulk_output_token_cap:
   ~20-30 tokens (compact structured JSON: label + evidence_ids only) for bulk experiments; verbose explanations reserved for explanation-quality-focused experiments only

H_experiments_that_should_be_local_only:
   - E00 (done)
   - E04 rule baseline
   - E06 retrieval optimisation (no LLM calls)
   - E09 agent justification analysis (reuses existing predictions)
   - the bulk of E12/E14 architecture comparison (local-first per the reconstruction brief

## Conclusion / decision

**The planned E01+E03+E15 programme (RECOMMENDED scenario) costs ~$0.75 of the $3.75 allowed
forward budget — budget is not the binding constraint.** The real constraints are: (1) local
runtime (~12.4 hours for a full 4-architecture local TEST pass), and (2) Gemini 2.5 Flash
Lite's retirement on 2026-10-16 (~3 weeks away), which is a genuine risk to any plan assuming
that model stays available through the whole hosted programme.

Decisions flagged for approval, not resolved here: the Gemini retirement risk, the exact
Oracle sample size (150 vs. 300), and whether Groq's free-tier hosting counts as one of the
"2 hosted" Oracle slots. Full write-up: `summary.md`.